# Batch controller runs (multiple levels & seeds)

Runs the same simulation loop as `run_controller.ipynb` for every **(level, seed)** pair and writes MP4 files to disk.

Each run saves one video per camera (`backcam`, `birdeyecam`) under `OUTPUT_DIR / level{L}_seed{S}/`.

MuJoCo renderers must be closed between runs (see `close_simulation`); otherwise later videos are black.

Edit the configuration cell below, then run all cells once.

In [1]:
%reload_ext autoreload
%autoreload 2

Failed to read module file 'C:\Users\ghugu\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\urllib\parse.py' for module 'urllib.parse': UnicodeDecodeError
Traceback (most recent call last):
  File "d:\controlling_behaviors_in_animals_and_robots\cobar-2026\Group13\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\controlling_behaviors_in_animals_and_robots\cobar-2026\Group13\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ghugu\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gc

## Configuration

In [2]:
from pathlib import Path
import random


# Simulation levels (0–4). See MiniprojectSimulation for what each enables.
LEVELS = [1, 2, 3, 4]
LEVELS = [4]

# Random seeds — each seed changes terrain, grass layout, banana position, wind, etc.

random_seeds = [random.randint(1, 1000) for _ in range(7)]

SEEDS = random_seeds + [1, 67, 777]
SEEDS = [127, 565, 239, 147, 809, 558, 800, 1, 67, 777]

# Timesteps per run (same default as run_controller.ipynb)
MAX_STEPS = 50_000

# Where to write videos (created if missing)
OUTPUT_DIR = Path("outputs/videos")

# Set True to stop early when the fly reaches the banana (within 3 units)
STOP_ON_GOAL = True

## Run all combinations

In [3]:
import gc
from itertools import product

import numpy as np
from flygym.compose import ActuatorType
from tqdm.auto import tqdm

from miniproject.simulation import MiniprojectSimulation
from miniproject.utils import close_simulation
from submission.controller import Controller


def reached_goal(sim: MiniprojectSimulation) -> bool:
    banana_xy = sim.world.banana_xy
    fly_xy = np.array(sim.get_body_positions(sim.fly.name)[0][:2])
    return np.linalg.norm(fly_xy - banana_xy) <= 3.0


def run_one(level: int, seed: int, output_dir: Path) -> dict:
    """Run simulation and save camera videos. Returns paths written."""
    sim = MiniprojectSimulation(level=level, seed=seed)
    controller = Controller(sim)
    run_dir = output_dir / f"level{level}_seed{seed}"
    saved: list[Path] = []
    try:
        for step in range(MAX_STEPS):
            if STOP_ON_GOAL and reached_goal(sim):
                break
            joint_angles, adhesion = controller.step(sim, step)
            sim.set_actuator_inputs(sim.fly.name, ActuatorType.POSITION, joint_angles)
            sim.set_actuator_inputs(sim.fly.name, ActuatorType.ADHESION, adhesion)
            sim.step()
            sim.render_as_needed()

        run_dir.mkdir(parents=True, exist_ok=True)
        sim.renderer.save_video(run_dir)
        saved = sorted(run_dir.glob("*.mp4"))
    finally:
        del controller
        close_simulation(sim)
        del sim
        gc.collect()
    return {"level": level, "seed": seed, "dir": run_dir, "videos": saved}


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
combos = list(product(LEVELS, SEEDS))
print(f"{len(combos)} runs: levels {LEVELS}, seeds {SEEDS}")
print(f"Output root: {OUTPUT_DIR.resolve()}")

results = []
for level, seed in tqdm(combos, desc="batch"):
    info = run_one(level, seed, OUTPUT_DIR)
    results.append(info)
    names = ", ".join(p.name for p in info["videos"])
    tqdm.write(f"level={level} seed={seed} -> {info['dir']} ({names})")

10 runs: levels [4], seeds [127, 565, 239, 147, 809, 558, 800, 1, 67, 777]
Output root: D:\controlling_behaviors_in_animals_and_robots\cobar-2026\Group13\miniproject\outputs\videos


batch:   0%|          | 0/10 [00:00<?, ?it/s]

level=4 seed=127 -> outputs\videos\level4_seed127 (birdeyecam.mp4, nmf-backcam.mp4)
level=4 seed=565 -> outputs\videos\level4_seed565 (birdeyecam.mp4, nmf-backcam.mp4)
level=4 seed=239 -> outputs\videos\level4_seed239 (birdeyecam.mp4, nmf-backcam.mp4)
level=4 seed=147 -> outputs\videos\level4_seed147 (birdeyecam.mp4, nmf-backcam.mp4)
level=4 seed=809 -> outputs\videos\level4_seed809 (birdeyecam.mp4, nmf-backcam.mp4)
level=4 seed=558 -> outputs\videos\level4_seed558 (birdeyecam.mp4, nmf-backcam.mp4)
level=4 seed=800 -> outputs\videos\level4_seed800 (birdeyecam.mp4, nmf-backcam.mp4)
level=4 seed=1 -> outputs\videos\level4_seed1 (birdeyecam.mp4, nmf-backcam.mp4)
level=4 seed=67 -> outputs\videos\level4_seed67 (birdeyecam.mp4, nmf-backcam.mp4)
level=4 seed=777 -> outputs\videos\level4_seed777 (birdeyecam.mp4, nmf-backcam.mp4)


## Summary

In [4]:
print(f"Finished {len(results)} runs.\n")
for r in results:
    for video in r["videos"]:
        print(video.resolve())

ERROR:autoreload:Failed to read module file 'd:\controlling_behaviors_in_animals_and_robots\cobar-2026\Group13\miniproject\submission\controller.py' for module 'submission.controller': UnicodeDecodeError
Traceback (most recent call last):
  File "d:\controlling_behaviors_in_animals_and_robots\cobar-2026\Group13\.venv\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 219, in update_sources
    self.source_by_modname[new_modname] = f.read()
                                          ^^^^^^^^
  File "C:\Users\ghugu\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 1261: character maps to <undefined>


Finished 10 runs.

D:\controlling_behaviors_in_animals_and_robots\cobar-2026\Group13\miniproject\outputs\videos\level4_seed127\birdeyecam.mp4
D:\controlling_behaviors_in_animals_and_robots\cobar-2026\Group13\miniproject\outputs\videos\level4_seed127\nmf-backcam.mp4
D:\controlling_behaviors_in_animals_and_robots\cobar-2026\Group13\miniproject\outputs\videos\level4_seed565\birdeyecam.mp4
D:\controlling_behaviors_in_animals_and_robots\cobar-2026\Group13\miniproject\outputs\videos\level4_seed565\nmf-backcam.mp4
D:\controlling_behaviors_in_animals_and_robots\cobar-2026\Group13\miniproject\outputs\videos\level4_seed239\birdeyecam.mp4
D:\controlling_behaviors_in_animals_and_robots\cobar-2026\Group13\miniproject\outputs\videos\level4_seed239\nmf-backcam.mp4
D:\controlling_behaviors_in_animals_and_robots\cobar-2026\Group13\miniproject\outputs\videos\level4_seed147\birdeyecam.mp4
D:\controlling_behaviors_in_animals_and_robots\cobar-2026\Group13\miniproject\outputs\videos\level4_seed147\nmf-backc